# Structured & SQL RAG — Panduan untuk Pemula

**Patria & Co. 2026** · [www.patriaco.co.uk](https://www.patriaco.co.uk)

Banyak pengetahuan perusahaan tersimpan di dalam **database**, bukan di dalam dokumen.
Notebook ini mengajarkan pola: **ambil skema → tulis SQL → jalankan → jawab**.

## Kenapa RAG dokumen biasa tidak cukup?

RAG (*Retrieval-Augmented Generation*) versi dokumen bekerja begini: potong dokumen jadi
paragraf → ubah jadi angka (*embedding*) → cari paragraf paling mirip dengan pertanyaan →
model membaca paragraf itu untuk menjawab. Ini bagus **kalau jawabannya memang sudah
tertulis** di suatu tempat.

Tapi coba pertanyaan ini: *"Berapa total pendapatan dari pesanan yang sudah dikirim?"*
Tidak ada satu paragraf pun yang menyimpan angka itu. Angkanya baru muncul setelah kita:

1. menggabungkan (JOIN) tabel `orders` dengan tabel `products`,
2. menyaring hanya baris dengan status `shipped`,
3. mengalikan `harga × jumlah` tiap baris,
4. menjumlahkan semuanya.

Pencarian berbasis kemiripan teks itu jago mencari **kalimat yang mirip**, tapi dia
**tidak bisa berhitung**. Karena itu alurnya kita ganti:

| RAG untuk dokumen | RAG untuk database (SQL RAG) |
|---|---|
| embed → retrieve → ground → generate | **retrieve skema → generate SQL → execute → answer** |

## Apa yang akan kita bangun

1. Database **SQLite** kecil dengan 3 tabel: `customers`, `products`, `orders`.
2. **Schema card** — kartu ringkas berisi info tiap tabel.
3. **Glosarium bisnis** — kamus istilah seperti "revenue" → rumus SQL-nya.
4. `retrieve_schema()` — memilih tabel yang relevan dengan pertanyaan.
5. `generate_sql()` — mengubah pertanyaan bahasa manusia jadi query SQL.
6. `sql_rag()` — pipeline lengkap: gabungkan semua langkah di atas.
7. `route()` — penentu: pertanyaan ini sebaiknya dijawab pakai SQL atau dokumen?

> **Semua berjalan offline**, tidak perlu internet atau API key. SQL-nya **sungguhan
> dijalankan** di database SQLite. Dua bagian sengaja kita sederhanakan supaya notebook
> ringan: (1) pemilihan tabel pakai skor kata sederhana (bukan model embedding), dan
> (2) SQL dibuat dengan aturan/pola teks (bukan LLM). Alur besarnya tetap sama seperti
> sistem produksi sungguhan.


## Langkah 0 — Persiapan (Setup)

Kita hanya butuh 2 modul bawaan Python (tidak perlu `pip install` apa pun):

- `re` → memecah kalimat menjadi kata-kata (disebut *tokenizing*).
- `sqlite3` → database sungguhan yang jalan di memori komputer, otomatis tersedia di Python.


In [ ]:
import re
import sqlite3

print(f"SQLite versi {sqlite3.sqlite_version} siap dipakai.")
print("Tidak perlu internet. Tidak perlu API key.")


## Langkah 1 — Membuat database contoh

Ini contoh data yang biasa hidup di **database** toko online, bukan di dokumen teks:

- **`customers`** — 1 baris = 1 pelanggan. Kolom `tier` berisi `free`, `pro`, atau `enterprise`.
- **`products`** — 1 baris = 1 produk, dengan `price` = harga satuan.
- **`orders`** — 1 baris = 1 pesanan. Menghubungkan pelanggan ↔ produk, plus `quantity`
  (jumlah beli) dan `status` (`shipped`/`refunded`/`pending`).

> ⚠️ **Perhatikan baik-baik:** tidak ada kolom bernama `revenue` di tabel manapun!
> Pendapatan harus **dihitung** dari `price × quantity`. Inilah alasan utama kita
> butuh SQL, bukan sekadar pencarian teks.


In [ ]:
def buat_database():
    """Membuat database SQLite di memori dan mengisinya dengan data contoh."""
    db = sqlite3.connect(":memory:")

    # --- 1. Buat struktur tabel (skema) ---
    db.executescript("""
        CREATE TABLE customers (
            id      INTEGER PRIMARY KEY,
            name    TEXT,
            country TEXT,
            tier    TEXT      -- 'free' | 'pro' | 'enterprise'
        );
        CREATE TABLE products (
            id       INTEGER PRIMARY KEY,
            name     TEXT,
            category TEXT,
            price    REAL
        );
        CREATE TABLE orders (
            id          INTEGER PRIMARY KEY,
            customer_id INTEGER,
            product_id  INTEGER,
            quantity    INTEGER,
            status      TEXT,     -- 'shipped' | 'refunded' | 'pending'
            order_date  TEXT      -- format: YYYY-MM-DD
        );
    """)

    # --- 2. Isi data pelanggan ---
    db.executemany("INSERT INTO customers VALUES (?,?,?,?)", [
        (1, "Ada",    "DE", "pro"),
        (2, "Bashir", "TR", "enterprise"),
        (3, "Chen",   "SG", "free"),
        (4, "Dilan",  "TR", "pro"),
        (5, "Eka",    "ID", "enterprise"),
        (6, "Farid",  "ID", "free"),
    ])

    # --- 3. Isi data produk ---
    db.executemany("INSERT INTO products VALUES (?,?,?,?)", [
        (10, "Aurora Lamp",    "lighting", 49.0),
        (11, "Nimbus Speaker", "audio",    129.0),
        (12, "Coil Cable",     "audio",    9.0),
        (13, "Solaris Bulb",   "lighting", 19.0),
    ])

    # --- 4. Isi data pesanan ---
    db.executemany("INSERT INTO orders VALUES (?,?,?,?,?,?)", [
        (100, 1, 11, 2, "shipped",  "2026-03-04"),
        (101, 2, 10, 1, "shipped",  "2026-03-09"),
        (102, 2, 12, 5, "refunded", "2026-03-12"),
        (103, 4, 11, 1, "shipped",  "2026-03-20"),
        (104, 3, 10, 3, "pending",  "2026-03-28"),
        (105, 5, 13, 4, "shipped",  "2026-04-02"),
        (106, 5, 11, 1, "shipped",  "2026-04-10"),
        (107, 6, 12, 2, "pending",  "2026-04-15"),
    ])

    db.commit()
    return db


db = buat_database()

print("Jumlah baris per tabel:")
print("  customers =", db.execute("SELECT COUNT(*) FROM customers").fetchone()[0])
print("  products  =", db.execute("SELECT COUNT(*) FROM products").fetchone()[0])
print("  orders    =", db.execute("SELECT COUNT(*) FROM orders").fetchone()[0])
print()
print("Ingat: kolom 'revenue' TIDAK ADA — harus dihitung dari price * quantity.")


## Langkah 2 — Schema Card dan Glosarium Bisnis

**Schema card** = ringkasan singkat 1 tabel: nama tabel, daftar kolom, dan penjelasan
1-2 kalimat tentang isi tabelnya. Kartu inilah yang nanti **dicari (retrieve)** —
bukan dokumen, bukan paragraf. Di sistem produksi sungguhan, kartu ini diubah jadi
*embedding* oleh model bahasa.

**Glosarium bisnis** = kamus "istilah bisnis → kolom database". Pengguna sering bertanya
pakai kata seperti "revenue", "pelanggan aktif", "churn" — kata-kata ini **bukan** nama
kolom asli. Tanpa glosarium, model bisa saja mengarang nama kolom yang tidak ada.
Dengan glosarium, model tahu bahwa "revenue" = `SUM(price × quantity)`.


In [ ]:
# --- Schema card: 1 kartu per tabel ---
SCHEMA_CARDS = {
    "customers": {
        "columns": ["id", "name", "country", "tier"],
        "doc": "1 baris = 1 pelanggan: nama, kode negara, dan tier langganan "
               "(free, pro, enterprise). Dipakai untuk pertanyaan jumlah pelanggan.",
    },
    "products": {
        "columns": ["id", "name", "category", "price"],
        "doc": "1 baris = 1 produk: nama, kategori, dan harga satuan (price). "
               "Revenue dihitung dari price di sini dikali quantity di tabel orders.",
    },
    "orders": {
        "columns": ["id", "customer_id", "product_id", "quantity", "status", "order_date"],
        "doc": "1 baris = 1 pesanan: pelanggan mana beli produk apa, berapa banyak "
               "(quantity), status (shipped/refunded/pending), dan tanggalnya.",
    },
}

# --- Glosarium: istilah bisnis -> arti sesungguhnya di database ---
GLOSARIUM = {
    "revenue":             "SUM(products.price * orders.quantity) — tidak ada kolom 'revenue'",
    "pelanggan enterprise": "customers.tier = 'enterprise'",
    "refund / dikembalikan": "orders.status = 'refunded'",
    "shipped / terkirim":   "orders.status = 'shipped'",
    "pending / tertunda":   "orders.status = 'pending'",
}

print("=== SCHEMA CARDS ===")
for tabel, kartu in SCHEMA_CARDS.items():
    print(f"\n[{tabel}] kolom: {kartu['columns']}")
    print(f"  -> {kartu['doc']}")

print("\n=== GLOSARIUM BISNIS ===")
for istilah, arti in GLOSARIUM.items():
    print(f"  '{istilah}' = {arti}")


## Langkah 3 — Mengubah kartu jadi teks yang bisa dibandingkan

Sebelum bisa memilih tabel mana yang relevan, kita perlu ubah tiap kartu jadi
**teks tunggal**: gabungan nama tabel + daftar kolom + deskripsi. Teks inilah yang
nanti dibandingkan dengan pertanyaan pengguna.

`ambil_kata()` mengubah kalimat jadi kumpulan kata dalam huruf kecil — ini bahan
bakar untuk skor kemiripan sederhana di langkah berikutnya.


In [ ]:
def ambil_kata(teks):
    """Mengambil semua kata/angka dari sebuah teks, huruf kecil semua.

    Contoh: 'Total Revenue?' -> {'total', 'revenue'}
    """
    return set(re.findall(r"[a-z0-9]+", teks.lower()))


def teks_kartu(nama_tabel, kartu):
    """Gabungkan nama tabel + kolom + deskripsi jadi satu teks."""
    return nama_tabel + " " + " ".join(kartu["columns"]) + " " + kartu["doc"]


# Contoh: lihat teks gabungan untuk kartu 'products'
contoh_teks = teks_kartu("products", SCHEMA_CARDS["products"])
print("Teks gabungan kartu 'products':")
print(f"  {contoh_teks}")
print()
print("Kata-kata unik di dalamnya:")
print(f"  {sorted(ambil_kata(contoh_teks))}")


## Langkah 4 — Menghitung skor kemiripan kata

Kita pakai cara paling sederhana: **hitung berapa banyak kata yang sama** antara
pertanyaan dan teks kartu. Makin banyak kata yang cocok, makin tinggi skornya.

> Di sistem produksi sungguhan, cara ini biasanya diganti dengan **model embedding**
> (misalnya `sentence-transformers`) yang mengukur kemiripan *makna*, bukan sekadar
> kata yang sama persis. Tapi untuk belajar konsepnya, cara sederhana ini sudah cukup
> — dan untuk contoh-contoh di notebook ini, **hasil akhirnya tetap sama**.


In [ ]:
def skor_kartu(pertanyaan, nama_tabel, kartu):
    """Skor = banyaknya kata yang sama antara pertanyaan dan teks kartu.

    Contoh:
        pertanyaan = 'total revenue shipped orders'
        teks kartu 'orders' mengandung kata 'orders' dan 'shipped'
        -> skornya 2
    """
    kata_pertanyaan = ambil_kata(pertanyaan)
    kata_kartu = ambil_kata(teks_kartu(nama_tabel, kartu))
    return len(kata_pertanyaan & kata_kartu)


# Coba hitung skor tiap tabel untuk pertanyaan tentang revenue
pertanyaan = "What was our total revenue from shipped orders?"
print(f"Pertanyaan: '{pertanyaan}'\n")
print("Skor per tabel:")
for tabel, kartu in SCHEMA_CARDS.items():
    print(f"  {tabel:<10} -> skor = {skor_kartu(pertanyaan, tabel, kartu)}")


## Langkah 5 — Retrieve Schema: ambil tabel yang relevan saja

Inilah langkah **schema linking**. Kita hitung skor semua tabel, lalu ambil
**top-k** (k tabel dengan skor tertinggi). Kenapa tidak ambil semua tabel saja?

- Database sungguhan bisa punya **ratusan tabel** dan ribuan kolom. Menempelkan
  seluruh skema ke prompt itu mahal (boros token) dan bisa membingungkan model.
- Kemampuan LLM bernalar dengan tabel akan **menurun** seiring makin banyak tabel
  yang ditempelkan — bahkan sebelum batas konteks (context window) tercapai.

Untuk pertanyaan revenue, `k=2` sudah cukup: kita butuh `orders` (untuk quantity &
status) dan `products` (untuk price) supaya bisa JOIN + SUM.


In [ ]:
def retrieve_schema(pertanyaan, k=2):
    """Beri skor ke semua kartu skema, kembalikan k kartu dengan skor tertinggi.

    Return: list berisi (nama_tabel, skor, kartu), diurutkan dari skor tertinggi.
    """
    hasil = []
    for tabel, kartu in SCHEMA_CARDS.items():
        hasil.append((tabel, skor_kartu(pertanyaan, tabel, kartu), kartu))
    hasil.sort(key=lambda item: -item[1])   # urutkan skor tinggi ke rendah
    return hasil[:k]


def tampilkan_schema(hasil_retrieve):
    """Format skema terpilih menjadi teks yang mudah dibaca."""
    baris = []
    for tabel, _skor, kartu in hasil_retrieve:
        kolom = ", ".join(kartu["columns"])
        baris.append(f"TABLE {tabel}({kolom})  -- {kartu['doc']}")
    return "\n".join(baris)


pertanyaan = "What was our total revenue from shipped orders?"
hasil = retrieve_schema(pertanyaan, k=2)

print(f"Pertanyaan: '{pertanyaan}'\n")
print("Tabel terpilih (top-2):", [tabel for tabel, _, _ in hasil])
print()
print("Skema yang diambil:")
print(tampilkan_schema(hasil))


## Langkah 6 — Generate SQL (versi aturan sederhana)

Di sistem produksi sungguhan, langkah ini mengirim prompt ke LLM berisi:
**(pertanyaan + skema terpilih + glosarium + contoh SQL)**, lalu mem-parsing SQL
yang dikembalikan model. Di notebook ini kita ganti LLM dengan **pencocokan pola
teks sederhana** (rule-based) — supaya tidak butuh model apa pun — tapi SQL yang
dihasilkan **tetap SQL sungguhan** yang benar-benar bisa dijalankan.

Dua hal penting untuk dipahami:

1. **Urutan aturan itu penting.** Aturan yang lebih spesifik (misalnya "produk
   terlaris") dicek **sebelum** aturan yang lebih umum ("total revenue"), supaya
   tidak salah tertangkap oleh aturan yang lebih umum.
2. **Guard clause (klausa penjaga).** Aturan revenue mensyaratkan tabel `orders`
   **dan** `products` sama-sama ada di skema yang diambil. Kalau salah satu tidak
   ada, fungsi **menolak** (mengembalikan `None`) — lebih baik menolak daripada
   menghasilkan angka yang salah.


In [ ]:
def generate_sql(pertanyaan, hasil_retrieve):
    """Mengubah pertanyaan bahasa manusia menjadi SQL, berdasarkan tabel yang tersedia.

    Memakai pola teks sederhana (bukan LLM). Mengembalikan None kalau tidak ada
    pola yang cocok, atau kalau tabel yang dibutuhkan belum tersedia.
    """
    q = pertanyaan.lower()
    tabel_tersedia = {tabel for tabel, _, _ in hasil_retrieve}

    # Aturan diurutkan dari yang PALING SPESIFIK ke yang paling umum.

    # 1) "produk terlaris / top product by revenue"
    if ("top" in q or "best selling" in q) and {"orders", "products"} <= tabel_tersedia:
        return ("SELECT p.name, SUM(p.price * o.quantity) AS revenue "
                "FROM orders o JOIN products p ON o.product_id = p.id "
                "WHERE o.status = 'shipped' "
                "GROUP BY p.name ORDER BY revenue DESC LIMIT 1;")

    # 2) "rata-rata nilai pesanan / average order value"
    if ("average" in q or "avg" in q) and "order" in q and {"orders", "products"} <= tabel_tersedia:
        return ("SELECT AVG(p.price * o.quantity) "
                "FROM orders o JOIN products p ON o.product_id = p.id "
                "WHERE o.status = 'shipped';")

    # 3) "total revenue"
    if "revenue" in q and {"orders", "products"} <= tabel_tersedia:
        if "shipped" in q:
            return ("SELECT SUM(p.price * o.quantity) "
                    "FROM orders o JOIN products p ON o.product_id = p.id "
                    "WHERE o.status = 'shipped';")
        return ("SELECT SUM(p.price * o.quantity) "
                "FROM orders o JOIN products p ON o.product_id = p.id;")

    # 4) "berapa pesanan yang di-refund"
    if "refund" in q and "orders" in tabel_tersedia:
        return "SELECT COUNT(*) FROM orders WHERE status = 'refunded';"

    # 5) "berapa pesanan bulan Maret/April"
    if ("how many" in q or "number of" in q) and "orders" in tabel_tersedia:
        if "march" in q or "maret" in q:
            return ("SELECT COUNT(*) FROM orders "
                    "WHERE order_date >= '2026-03-01' AND order_date < '2026-04-01';")
        if "april" in q:
            return ("SELECT COUNT(*) FROM orders "
                    "WHERE order_date >= '2026-04-01' AND order_date < '2026-05-01';")

    # 6) "berapa pelanggan <tier>"
    if "how many" in q and "customers" in tabel_tersedia:
        for tier in ("enterprise", "pro", "free"):
            if tier in q:
                return f"SELECT COUNT(*) FROM customers WHERE tier = '{tier}';"
        return "SELECT COUNT(*) FROM customers;"

    # Tidak ada pola yang cocok, atau tabelnya belum lengkap -> menolak.
    return None


# Coba hasilkan SQL untuk pertanyaan revenue
sql = generate_sql("What was our total revenue from shipped orders?", hasil)
print("SQL yang dihasilkan:")
print(f"  {sql}")


## Langkah 7 — Execute & Answer: gabungkan semuanya

Inilah bagian yang **tidak ada** di pipeline dokumen: kita benar-benar **menjalankan
SQL** di database SQLite. Database yang menghitung angkanya secara **pasti/eksak**,
jadi model tidak perlu menebak-nebak.

`sql_rag()` adalah pipeline lengkapnya:
**retrieve schema → generate SQL → execute → answer.**

Kalau `generate_sql()` menolak (mengembalikan `None`), kita sampaikan itu dengan
jelas ke pengguna — bukan memaksa menjalankan query yang salah.


In [ ]:
def jawaban_dari_baris(baris):
    """Ubah hasil query (list of tuples) jadi jawaban singkat yang mudah dibaca.

    Di sistem produksi, baris hasil biasanya dikirim balik ke LLM supaya
    dirangkai jadi kalimat natural. Di sini kita cukup format nilainya.
    """
    if not baris:
        return "Tidak ada baris yang cocok."
    if len(baris[0]) > 1:                    # hasil multi-kolom, misal top-N
        return " | ".join(str(v) for v in baris[0])
    return f"{baris[0][0]}"


def sql_rag(db, pertanyaan, k=2):
    """Pipeline SQL RAG lengkap.

    Args:
        db: koneksi sqlite3.
        pertanyaan: pertanyaan pengguna dalam bahasa natural.
        k: jumlah tabel yang diambil (default 2).

    Return: dict berisi tabel yang diambil, SQL yang dihasilkan, dan jawabannya.
    """
    # 1) Retrieve — pilih tabel yang relevan
    hasil_retrieve = retrieve_schema(pertanyaan, k=k)

    # 2) Generate — tulis SQL berdasarkan tabel yang terpilih
    sql = generate_sql(pertanyaan, hasil_retrieve)

    # 3) Kalau tidak ada SQL yang bisa dibuat, tolak dengan sopan
    if sql is None:
        return {
            "tables": [tabel for tabel, _, _ in hasil_retrieve],
            "sql": None,
            "answer": "Maaf, pertanyaan ini belum bisa diterjemahkan ke SQL.",
        }

    # 4) Execute — jalankan SQL sungguhan di database
    baris = db.execute(sql).fetchall()

    # 5) Answer — ubah hasil jadi jawaban
    return {
        "tables": [tabel for tabel, _, _ in hasil_retrieve],
        "sql": sql,
        "answer": jawaban_dari_baris(baris),
    }


# Uji coba end-to-end
hasil_akhir = sql_rag(db, "What was our total revenue from shipped orders?")
print("Tabel yang diambil :", hasil_akhir["tables"])
print("SQL                :", hasil_akhir["sql"])
print("Jawaban            :", hasil_akhir["answer"])
print()
print("Cek manual (pesanan shipped saja):")
print("  order 100: 129 x 2 = 258  (Nimbus Speaker)")
print("  order 101:  49 x 1 =  49  (Aurora Lamp)")
print("  order 103: 129 x 1 = 129  (Nimbus Speaker)")
print("  order 105:  19 x 4 =  76  (Solaris Bulb)")
print("  order 106: 129 x 1 = 129  (Nimbus Speaker)")
print("  TOTAL     = 641.0  <- harus sama dengan 'Jawaban' di atas")


## Langkah 8 — Router: SQL atau dokumen?

Sekarang bayangkan kita punya **dua jenis pipeline**: pipeline dokumen (RAG teks biasa)
dan pipeline SQL (yang baru kita bangun). Kita butuh **pengarah (router)** yang
menentukan setiap pertanyaan sebaiknya lewat jalur mana.

Aturan sederhana: kalau pertanyaan mengandung **kata sinyal angka/agregasi**
(`how many`, `count`, `total`, `average`, `top`, `revenue`, dsb.), arahkan ke **SQL**.
Sisanya (misalnya *"apa kebijakan refund kami?"*) diarahkan ke pipeline **dokumen**.

> ⚠️ **Yang paling berbahaya adalah salah arah secara diam-diam** — pertanyaan
> angka yang malah masuk ke pipeline dokumen. Model akan menemukan paragraf yang
> *terdengar* mirip lalu **mengarang angka**. Karena itu, kalau ragu, **condongkan
> ke SQL**: query yang salah biasanya gagal dengan jelas (error atau ditolak),
> sedangkan angka karangan terlihat meyakinkan padahal salah.


In [ ]:
SINYAL_SQL = (
    "how many", "count", "total", "sum", "average", "avg",
    "most", "top", "revenue", "number of",
    "berapa", "jumlah", "rata-rata",   # sinyal Bahasa Indonesia
)


def route(pertanyaan):
    """Kembalikan 'sql' kalau ada sinyal angka/agregasi, selain itu 'documents'."""
    q = pertanyaan.lower()
    if any(sinyal in q for sinyal in SINYAL_SQL):
        return "sql"
    return "documents"


contoh_pertanyaan = [
    "How many enterprise customers do we have?",
    "What is your refund policy?",
    "Berapa jumlah pelanggan enterprise?",
    "Apa syarat garansi produk?",
]
for q in contoh_pertanyaan:
    print(f"  route = {route(q):<9} <- {q}")


## Langkah 9 — Demo: banyak pertanyaan, dua pipeline

Sekarang kita jalankan beberapa pertanyaan lewat router. Pertanyaan yang bersifat
angka/agregasi akan masuk ke pipeline SQL (retrieve skema → generate SQL → execute
→ answer). Pertanyaan deskriptif akan diarahkan ke pipeline dokumen — di sini
sekadar contoh placeholder karena fokus notebook ini adalah bagian SQL.

Di akhir, kita **verifikasi otomatis** bahwa semua jawaban SQL memang sesuai
dengan data yang kita isi sendiri di Langkah 1 — supaya kita yakin pipeline-nya
berjalan benar, bukan cuma "terlihat" berjalan.


In [ ]:
DAFTAR_DEMO = [
    "How many enterprise customers do we have?",
    "What was our total revenue from shipped orders?",
    "How many orders were refunded?",
    "What is the average order value for shipped orders?",
    "What is the top product by revenue?",
    "How many orders were placed in March?",
    "What is your refund policy?",   # akan diarahkan ke pipeline dokumen
]

for q in DAFTAR_DEMO:
    arah = route(q)
    print(f"Q: {q}")
    print(f"   route = {arah}")
    if arah != "sql":
        print("   -> pipeline dokumen: retrieve paragraf, ground, generate jawaban.")
        print()
        continue
    hasil = sql_rag(db, q)
    print(f"   skema diambil : {hasil['tables']}")
    print(f"   SQL           : {hasil['sql']}")
    print(f"   jawaban       : {hasil['answer']}")
    print()

# --- Verifikasi otomatis: apakah jawaban SQL sesuai data yang kita seed? ---
jawaban_seharusnya = {
    "How many enterprise customers do we have?": "2",   # Bashir, Eka
    "What was our total revenue from shipped orders?": "641.0",
    "How many orders were refunded?": "1",               # order 102
    "How many orders were placed in March?": "5",        # order 100-104
}
print("=== Hasil Verifikasi ===")
semua_benar = True
for q, seharusnya in jawaban_seharusnya.items():
    jawaban_aktual = sql_rag(db, q)["answer"]
    benar = (jawaban_aktual == seharusnya)
    semua_benar = semua_benar and benar
    status = "OK" if benar else f"SALAH (dapat: {jawaban_aktual})"
    print(f"  [{status}] {q}  (seharusnya: {seharusnya})")

print()
print("SEMUA BENAR ✔" if semua_benar else "ADA YANG SALAH ✘ — cek ulang kode di atas!")
assert semua_benar, "Verifikasi gagal — periksa kembali fungsi generate_sql / sql_rag."


## Langkah 10 — Eksperimen: apa yang terjadi kalau skema kurang lengkap?

Bagian ini paling mudah dipahami lewat contoh **kegagalan**. `retrieve_schema()`
mengambil top-`k` tabel. Sekarang kita turunkan `k` jadi **1** dan ulangi pertanyaan
revenue. Yang terambil cuma tabel `orders`; tabel `products` (yang menyimpan `price`)
tidak ikut terambil.

Guard clause yang kita tulis di Langkah 6 akan **menolak** memberi jawaban — bukan
menebak angka yang salah. Inilah **kegagalan yang benar**: query itu butuh dua
tabel tapi cuma dapat satu, jadi lebih baik menolak.

> **Pelajaran penting:** pada data terstruktur, sebagian besar jawaban yang salah
> berasal dari **kegagalan schema linking** (tabel yang relevan tidak ikut terambil),
> bukan dari kesalahan menulis SQL itu sendiri.


In [ ]:
hasil_k1 = sql_rag(db, "What was our total revenue from shipped orders?", k=1)

print("k=1, tabel yang diambil :", hasil_k1["tables"])
print("k=1, SQL                :", hasil_k1["sql"])
print("k=1, jawaban            :", hasil_k1["answer"])
print()
print("Sistem menolak menjawab — bukan menebak. Inilah perilaku yang benar.")


## Langkah 11 — Perluasan: kueri yang lebih kompleks

Pipeline yang sama ternyata bisa menjawab jenis pertanyaan yang lebih rumit —
selama polanya sudah didukung `generate_sql()`. Kita coba:

- **AVG** → rata-rata nilai pesanan.
- **GROUP BY + ORDER BY + LIMIT** → produk dengan revenue tertinggi.
- **Filter tanggal** → jumlah pesanan pada bulan tertentu.

Tidak perlu kode tambahan — cukup panggil `sql_rag()` seperti biasa.


In [ ]:
pertanyaan_lanjutan = [
    "What is the average order value for shipped orders?",
    "What is the top product by revenue?",
    "How many orders were placed in April?",
    "How many orders were placed in March?",
]

for q in pertanyaan_lanjutan:
    hasil = sql_rag(db, q)
    print(f"Q: {q}")
    print(f"   tabel   : {hasil['tables']}")
    print(f"   SQL     : {hasil['sql']}")
    print(f"   jawaban : {hasil['answer']}")
    print()


## Langkah 12 — Perluasan: menambah tabel & pola pertanyaan baru

Bagian ini menunjukkan bagaimana pipeline **berkembang** seiring database bertambah
besar — persis seperti yang akan Anda lakukan di proyek nyata.

Kita tambahkan 1 tabel baru: **`reviews`** (ulasan produk, dengan `rating` 1-5).
Lalu kita:

1. Tambahkan **schema card**-nya ke `SCHEMA_CARDS`.
2. Tambahkan **pola SQL baru** di `generate_sql()` untuk pertanyaan
   *"What is the average rating for <produk>?"*.
3. Buktikan bahwa `retrieve_schema()` otomatis ikut mempertimbangkan tabel baru ini
   — kita tidak perlu mengubah fungsi retrieve sama sekali!


In [ ]:
# --- 1. Tambah tabel 'reviews' ke database ---
db.executescript("""
    CREATE TABLE reviews (
        id         INTEGER PRIMARY KEY,
        product_id INTEGER,
        rating     INTEGER,   -- 1 sampai 5
        comment    TEXT
    );
""")
db.executemany("INSERT INTO reviews VALUES (?,?,?,?)", [
    (1, 11, 5, "Suara jernih, sangat puas"),
    (2, 11, 4, "Bagus tapi baterainya cepat habis"),
    (3, 10, 3, "Lampu oke, kabelnya agak pendek"),
    (4, 13, 5, "Terang dan hemat listrik"),
    (5, 12, 2, "Kabel gampang putus"),
])
db.commit()
print("Tabel 'reviews' berhasil ditambahkan, jumlah baris:",
      db.execute("SELECT COUNT(*) FROM reviews").fetchone()[0])

# --- 2. Tambah schema card untuk 'reviews' ---
SCHEMA_CARDS["reviews"] = {
    "columns": ["id", "product_id", "rating", "comment"],
    "doc": "1 baris = 1 ulasan produk: rating 1-5 dan komentar pelanggan. "
           "Dipakai untuk pertanyaan tentang rating/ulasan produk.",
}

print()
print("SCHEMA_CARDS sekarang punya", len(SCHEMA_CARDS), "tabel:", list(SCHEMA_CARDS.keys()))


In [ ]:
# --- 3. Tambah pola SQL baru: rata-rata rating sebuah produk ---
def generate_sql_v2(pertanyaan, hasil_retrieve):
    """Versi generate_sql() yang diperluas: bisa jawab pertanyaan tentang rating.

    Kita tidak menimpa generate_sql() lama, supaya perbandingan "sebelum vs
    sesudah" tetap terlihat jelas.
    """
    q = pertanyaan.lower()
    tabel_tersedia = {tabel for tabel, _, _ in hasil_retrieve}

    # Pola baru: "average rating" / "rata-rata rating"
    if ("rating" in q) and ("average" in q or "rata-rata" in q) and \
       {"reviews", "products"} <= tabel_tersedia:
        for nama_produk in ("nimbus", "aurora", "solaris", "coil"):
            if nama_produk in q:
                return ("SELECT AVG(r.rating) FROM reviews r "
                        "JOIN products p ON r.product_id = p.id "
                        f"WHERE LOWER(p.name) LIKE '%{nama_produk}%';")
        return "SELECT AVG(rating) FROM reviews;"

    # Kalau bukan pola baru, pakai pola-pola lama seperti biasa
    return generate_sql(pertanyaan, hasil_retrieve)


def sql_rag_v2(db, pertanyaan, k=2):
    """sql_rag() versi diperluas, memakai generate_sql_v2()."""
    hasil_retrieve = retrieve_schema(pertanyaan, k=k)
    sql = generate_sql_v2(pertanyaan, hasil_retrieve)
    if sql is None:
        return {
            "tables": [tabel for tabel, _, _ in hasil_retrieve],
            "sql": None,
            "answer": "Maaf, pertanyaan ini belum bisa diterjemahkan ke SQL.",
        }
    baris = db.execute(sql).fetchall()
    return {
        "tables": [tabel for tabel, _, _ in hasil_retrieve],
        "sql": sql,
        "answer": jawaban_dari_baris(baris),
    }


# Uji coba pertanyaan baru tentang rating.
# Catatan: sekarang ada 4 tabel, dan pola ini butuh 'reviews' + 'products'
# sekaligus. Kita naikkan k jadi 3 supaya kans keduanya ikut ter-retrieve
# lebih besar (dengan skor kata sederhana, dua tabel itu belum tentu selalu
# jadi top-2 -- salah satu keterbatasan yang kita bahas juga di Langkah 10).
pertanyaan_rating = "What is the average rating for Nimbus Speaker?"
hasil = sql_rag_v2(db, pertanyaan_rating, k=3)
print(f"Pertanyaan: {pertanyaan_rating}")
print(f"  tabel   : {hasil['tables']}")
print(f"  SQL     : {hasil['sql']}")
print(f"  jawaban : {hasil['answer']}")
print()
print("Cek manual: rating Nimbus Speaker = (5 + 4) / 2 = 4.5")


## Langkah 13 — Menjelajahi isi tabel secara langsung

Kadang berguna untuk melihat langsung isi database — misalnya untuk memeriksa
apakah jawaban SQL kita masuk akal. Fungsi `tampilkan_tabel()` di bawah menampilkan
seluruh isi tabel apa pun dalam format rapi.


In [ ]:
def tampilkan_tabel(db, nama_tabel, batas=20):
    """Tampilkan isi sebuah tabel dalam bentuk kolom yang rapi."""
    info_kolom = db.execute(f"PRAGMA table_info({nama_tabel})").fetchall()
    nama_kolom = [k[1] for k in info_kolom]

    baris = db.execute(f"SELECT * FROM {nama_tabel} LIMIT {batas}").fetchall()

    header = " | ".join(f"{k:<12}" for k in nama_kolom)
    print(header)
    print("-" * len(header))
    for r in baris:
        print(" | ".join(f"{str(v):<12}" for v in r))


for nama_tabel in ["customers", "products", "orders", "reviews"]:
    print(f"\n=== {nama_tabel.upper()} ===")
    tampilkan_tabel(db, nama_tabel)


## Langkah 14 — Sekarang coba pertanyaan Anda sendiri!

Ubah isi variabel `pertanyaan_anda` di bawah, lalu jalankan cell-nya. Beberapa
contoh pola yang sudah didukung:

- *"How many free customers?"*
- *"How many orders were placed in April?"*
- *"What is the top product by revenue?"*
- *"What is the average order value for shipped orders?"*
- *"What is the average rating for Aurora Lamp?"* (pakai `sql_rag_v2`)

Atau coba pertanyaan yang **belum didukung** dan lihat bagaimana sistem menolak
dengan sopan — inilah perilaku yang tepat untuk sistem produksi (lebih baik
menolak daripada mengarang jawaban).


In [ ]:
pertanyaan_anda = "How many pro customers do we have?"

hasil = sql_rag_v2(db, pertanyaan_anda)
print(f"Pertanyaan          : {pertanyaan_anda}")
print(f"Tabel yang diambil  : {hasil['tables']}")
print(f"SQL yang dihasilkan : {hasil['sql']}")
print(f"Jawaban             : {hasil['answer']}")


## Ringkasan — Apa yang sudah Anda pelajari

1. **Pencarian teks biasa tidak bisa menjawab pertanyaan yang butuh perhitungan.**
   Angka seperti "total revenue" tidak tertulis di paragraf mana pun — dia baru ada
   setelah sebuah query menghitungnya. Mengambil paragraf yang "paling mirip" hanya
   akan menghasilkan **angka karangan**. Solusinya adalah query, bukan model
   embedding yang lebih canggih.

2. **Text-to-SQL RAG** menukar alur RAG dokumen (**embed → retrieve → ground →
   generate**) menjadi (**retrieve skema → generate SQL → execute → answer**).
   Database yang melakukan perhitungan secara **pasti**, jadi model tidak perlu
   menebak.

3. **Yang di-*retrieve* bukan cuma nama tabel**, tapi **schema card** (kolom +
   deskripsi) plus **glosarium bisnis** (istilah seperti "revenue" → rumus SQL-nya).
   Kombinasi ini mencegah sebagian besar kesalahan seperti "kolom dikarang" atau
   "rumus salah".

4. **Ambil sebagian skema saja (schema linking)**, jangan seluruh katalog tabel.
   Alasannya: (a) skema sungguhan bisa lebih besar dari batas konteks model;
   (b) kemampuan model bernalar di atas tabel menurun seiring makin banyak tabel
   yang ditempelkan — jauh sebelum limit token tercapai.

5. **Router** mengarahkan pertanyaan angka/agregasi ke SQL, sisanya ke pipeline
   dokumen. Kalau ragu, condongkan ke SQL — salah arah ke dokumen gagal **diam-diam**
   (angka dikarang), sedangkan salah arah ke SQL biasanya gagal **dengan jelas**
   (error atau ditolak).

6. **Guard clause dan "menolak dengan sopan"** itu penting. Sistem yang menolak
   menjawab saat skemanya tidak lengkap jauh lebih aman daripada sistem yang
   memberi angka karangan. Pada data terstruktur, tidak ada jawaban lebih baik
   daripada jawaban yang salah.

7. **Pipeline ini mudah diperluas** — kita buktikan sendiri di Langkah 12: tambah
   1 tabel baru (`reviews`) dan 1 pola SQL baru, tanpa mengubah `retrieve_schema()`
   sama sekali.

8. **Semua berjalan offline.** Bagian eksekusi (SQL ke SQLite) itu sungguhan,
   sedangkan pemilihan skema (skor kata) dan pembuatan SQL (aturan/pola) sengaja
   disederhanakan agar tidak butuh dependensi tambahan. Alur besarnya tetap sama
   dengan sistem produksi.

### Langkah lanjutan kalau ingin membangun versi produksi

- Ganti skor kata sederhana dengan **model embedding** (misalnya
  `sentence-transformers`) untuk memilih schema card.
- Ganti `generate_sql()` berbasis aturan dengan **LLM** yang diberi prompt berisi:
  pertanyaan + schema card + glosarium + beberapa contoh SQL.
- Tambahkan **validasi query** (misalnya `EXPLAIN QUERY PLAN`, cek sintaks, batasi
  tabel yang boleh diakses oleh model).
- Tambahkan **verifikasi hasil** — LLM kedua yang mengecek kewajaran jawaban
  sebelum dikirim ke pengguna.
- Tambahkan **logging** untuk setiap SQL yang dihasilkan — penting untuk audit dan
  proses debugging.

---

**© Patria & Co. 2026** · Konsultan AI & Data Science
🌐 [www.patriaco.co.uk](https://www.patriaco.co.uk)
